## **Problem Statement**

### **Business Context**

An automobile dealership in Los Vegas specializes in selling luxury and non-luxury vehicles. They cater to diverse customer preferences with varying vehicle specifications, such as mileage, engine capacity, and seating capacity. However, the dealership faces significant challenges in maintaining consistency and efficiency across its pricing strategy due to reliance on manual processes and disconnected systems. Pricing evaluations are prone to errors, updates are delayed, and scaling operations are difficult as demand grows. These inefficiencies impact revenue and customer trust. Recognizing the need for a reliable and scalable solution, the dealership is seeking to implement a unified system that ensures seamless integration of data-driven pricing decisions, adaptability to changing market conditions, and operational efficiency.

### **Objective**

The dealership has hired you as an MLOps Engineer to design and implement an MLOps pipeline that automates the pricing workflow. This pipeline will encompass data cleaning, preprocessing, transformation, model building, training, evaluation, and registration with CI/CD capabilities to ensure continuous integration and delivery. Your role is to overcome challenges such as integrating disparate data sources, maintaining consistent model performance, and enabling scalable, automated updates to meet evolving business needs. The expected outcomes are a robust, automated system that improves pricing accuracy, operational efficiency, and scalability, driving increased profitability and customer satisfaction.

### **Data Description**

The dataset contains attributes of used cars sold in various locations. These attributes serve as key data points for CarOnSell's pricing model. The detailed attributes are:

- **Segment:** Describes the category of the vehicle, indicating whether it is a luxury or non-luxury segment.

- **Kilometers_Driven:** The total number of kilometers the vehicle has been driven.

- **Mileage:** The fuel efficiency of the vehicle, measured in kilometers per liter (km/l).

- **Engine:** The engine capacity of the vehicle, measured in cubic centimeters (cc). 

- **Power:** The power of the vehicle's engine, measured in brake horsepower (BHP). 

- **Seats:** The number of seats in the vehicle, can influence the vehicle's classification, usage, and pricing based on customer needs.

- **Price:** The price of the vehicle, listed in lakhs (units of 100,000), represents the cost to the consumer for purchasing the vehicle.

## **1. AzureML Environment Setup and Data Preparation**

### **1.1 Connect to Azure Machine Learning Workspace**

In [1]:
# Handle to the workspace
from azure.ai.ml import MLClient

# Authentication package
from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()

# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id="33a3b601-2ec7-496b-9642-a194dc909354",
    resource_group_name="Project-MLOPS",
    workspace_name="mlops",
)

In [2]:
# Verify the connection by retrieving workspace metadata
ws = ml_client.workspaces.get(ml_client.workspace_name)
print(f"Connected successfully to Workspace: {ws.name}")
print(f"Resource Group: {ws.resource_group}")
print(f"Location: {ws.location}")

Connected successfully to Workspace: mlops
Resource Group: Project-MLOPS
Location: eastus


In [3]:
# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id="6490c64b-602a-4887-b258-36064f4cb8d4",
    resource_group_name="default_resourse_group",
    workspace_name="demo_workspace",
)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


### **1.2 Set Up Compute Cluster**

In [5]:
# Reference the pre-created lab compute cluster directly by name
cpu_compute_target = "cpu-cluster"
print(f"Using compute target: {cpu_compute_target}")

Using compute target: cpu-cluster


### **1.3 Register Dataset as Data Asset**

In [7]:
# Reference the already registered AzureML Data Asset directly
data_asset_name = "used-cars-data"
data_asset_uri = "azureml:used-cars-data@latest"

print(f"Data asset reference set to: {data_asset_uri}")

Data asset reference set to: azureml:used-cars-data@latest


In [5]:
ml_client.data.create_or_update(data_asset)

Data({'path': 'azureml://subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourcegroups/default_resourse_group/workspaces/demo_workspace/datastores/workspaceblobstore/paths/LocalUpload/2be82e6311791c3eb0847ecab5279e37/used_cars.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'used-cars-data', 'description': 'A dataset of used cars for price prediction', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourceGroups/default_resourse_group/providers/Microsoft.MachineLearningServices/workspaces/demo_workspace/data/used-cars-data/versions/8', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/c002/code/Users/TESTP3XHV8C5OT_1734342789061', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7fc05c4dd840>, 'seria

### **1.4 Create and Configure Job Environment**

In [8]:
import os

# Create environment configuration directory
src_dir_env = "./env"
os.makedirs(src_dir_env, exist_ok=True)

In [9]:
%%writefile ./env/conda.yml
name: sklearn-env
channels:
  - conda-forge
dependencies:
  - python=3.8
  - pip=21.2.4
  - scikit-learn=0.23.2
  - scipy=1.7.1
  - pip:  
    - mlflow==2.8.1
    - azureml-mlflow==1.51.0
    - azureml-inference-server-http
    - azureml-core==1.49.0
    - cloudpickle==1.6.0

Writing ./env/conda.yml


In [10]:
# Reference the pre-registered environment directly
environment_name = "machine_learning_E2E@latest"
print(f"Environment reference set to: {environment_name}")

Environment reference set to: machine_learning_E2E@latest


## **2. Model Development Workflow**

### **2.1 Data Preparation**

This **Data Preparation job** is designed to process an input dataset by splitting it into two parts: one for training the model and the other for testing it. The script accepts three inputs: the location of the input data (`used_cars.csv`), the ratio for splitting the data into training and testing sets (`test_train_ratio`), and the paths to save the resulting training (`train_data`) and testing (`test_data`) data. The script first reads the input CSV data from a data asset URI, then splits it using Scikit-learn's train_test_split function, and saves the two parts to the specified directories. It also logs the number of records in both the training and testing datasets using MLflow.

In [11]:
    # ------- WRITE YOUR CODE HERE -------
import os

# Create directory for Python scripts
src_dir = "./src"
os.makedirs(src_dir, exist_ok=True)
print(f"Directory {src_dir} created successfully.")

Directory ./src created successfully.


In [13]:
%%writefile src/prep.py
import argparse
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import mlflow

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--raw_data", type=str, help="Path to raw data asset")
    parser.add_argument("--test_train_ratio", type=float, default=0.2, help="Train-test split ratio")
    parser.add_argument("--train_data", type=str, help="Directory to save train dataset")
    parser.add_argument("--test_data", type=str, help="Directory to save test dataset")
    args = parser.parse_args()

    mlflow.start_run()

    # Load raw dataset
    df = pd.read_csv(args.raw_data)

    # Train-test split
    train_df, test_df = train_test_split(df, test_size=args.test_train_ratio, random_state=42)

    # Ensure output folders exist
    os.makedirs(args.train_data, exist_ok=True)
    os.makedirs(args.test_data, exist_ok=True)

    # Save split datasets
    train_df.to_csv(os.path.join(args.train_data, "train.csv"), index=False)
    test_df.to_csv(os.path.join(args.test_data, "test.csv"), index=False)

    # Log metrics with MLflow
    mlflow.log_metric("train_records", len(train_df))
    mlflow.log_metric("test_records", len(test_df))

    mlflow.end_run()

if __name__ == "__main__":
    main()

Writing src/prep.py


In [14]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

prep_job = command(
    name="prep_data",
    display_name="Data Preparation Step",
    code="./src",
    command="python prep.py --raw_data ${{inputs.raw_data}} --test_train_ratio ${{inputs.test_train_ratio}} --train_data ${{outputs.train_data}} --test_data ${{outputs.test_data}}",
    environment="machine_learning_E2E@latest",
    inputs={
        "raw_data": Input(type=AssetTypes.URI_FILE, path="azureml:used-cars-data@latest"),
        "test_train_ratio": 0.2,
    },
    outputs={
        "train_data": Output(type=AssetTypes.URI_FOLDER),
        "test_data": Output(type=AssetTypes.URI_FOLDER),
    },
    compute=cpu_compute_target,
)

#### **Define Data Preparation job**

For this AzureML job, we define the `command` object that takes input files and output directories, then executes the script with the provided inputs and outputs. The job runs in a pre-configured AzureML environment with the necessary libraries. The result will be two separate datasets for training and testing, ready for use in subsequent steps of the machine learning pipeline.

In [ ]:
    # ------- WRITE YOUR CODE HERE -------

### **2.2 Training the Model**

This Model Training job is designed to train a **Random Forest Regressor** on the dataset that was split into training and testing sets in the previous data preparation job. This job script accepts five inputs: the path to the training data (`train_data`), the path to the testing data (`test_data`), the number of trees in the forest (`n_estimators`, with a default value of 100), the maximum depth of the trees (`max_depth`, which is set to None by default), and the path to save the trained model (`model_output`).

The script begins by reading the training and testing data files, then processes the data to separate features (X) and target labels (y). A Random Forest Regressor model is initialized using the given n_estimators and max_depth, and it is trained using the training data. The model's performance is evaluated using the `Mean Squared Error (MSE)`. The MSE score is logged in MLflow. Finally, the trained model is saved and stored in the specified output location as an MLflow model. The job completes by logging the final MSE score and ending the MLflow run.


In [15]:
%%writefile src/train.py
import argparse
import os
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import mlflow
import mlflow.sklearn

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_data", type=str, help="Path to train dataset directory")
    parser.add_argument("--test_data", type=str, help="Path to test dataset directory")
    parser.add_argument("--n_estimators", type=int, default=100, help="Number of trees")
    parser.add_argument("--max_depth", type=int, default=None, help="Tree max depth")
    parser.add_argument("--model_output", type=str, help="Path to save output model")
    args = parser.parse_args()

    mlflow.autolog()

    # Read train and test data
    train_df = pd.read_csv(os.path.join(args.train_data, "train.csv"))
    test_df = pd.read_csv(os.path.join(args.test_data, "test.csv"))

    # Separate target column ('price')
    X_train = train_df.drop(columns=["price"])
    y_train = train_df["price"]
    X_test = test_df.drop(columns=["price"])
    y_test = test_df["price"]

    # One-hot encode categorical features ('Segment')
    X_train = pd.get_dummies(X_train, drop_first=True)
    X_test = pd.get_dummies(X_test, drop_first=True)
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

    # Train Random Forest model
    model = RandomForestRegressor(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth if args.max_depth != 0 else None,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Evaluate Mean Squared Error
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mlflow.log_metric("MSE", mse)

    # Save trained MLflow model artifact
    os.makedirs(args.model_output, exist_ok=True)
    mlflow.sklearn.save_model(sk_model=model, path=args.model_output)

if __name__ == "__main__":
    main()

Writing src/train.py


#### **Define Model Training Job**

For this AzureML job, we define the `command` object that takes the paths to the training and testing data, the number of trees in the forest (`n_estimators`), and the maximum depth of the trees (`max_depth`) as inputs, and outputs the trained model. The command runs in a pre-configured AzureML environment with all the necessary libraries. The job produces a trained **Random Forest Regressor model**, which can be used for predicting the price of used cars based on the given attributes.

In [16]:
train_job = command(
    name="train_model",
    display_name="Model Training Step",
    code="./src",
    command="python train.py --train_data ${{inputs.train_data}} --test_data ${{inputs.test_data}} --n_estimators ${{inputs.n_estimators}} --max_depth ${{inputs.max_depth}} --model_output ${{outputs.model_output}}",
    environment="machine_learning_E2E@latest",
    inputs={
        "train_data": Input(type=AssetTypes.URI_FOLDER),
        "test_data": Input(type=AssetTypes.URI_FOLDER),
        "n_estimators": 100,
        "max_depth": 10,
    },
    outputs={
        "model_output": Output(type=AssetTypes.MLFLOW_MODEL),
    },
    compute=cpu_compute_target,
)

### **2.3 Registering the Best Trained Model**

The **Model Registration job** is designed to take the best-trained model from the hyperparameter tuning sweep job and register it in MLflow as a versioned artifact for future use in the used car price prediction pipeline. This job script accepts one input: the path to the trained model (model). The script begins by loading the model using the `mlflow.sklearn.load_model()` function. Afterward, it registers the model in the MLflow model registry, assigning it a descriptive name (`used_cars_price_prediction_model`) and specifying an artifact path (`random_forest_price_regressor`) where the model artifacts will be stored. Using MLflow's `log_model()` function, the model is logged along with its metadata, ensuring that the model is easily trackable and retrievable for future evaluation, deployment, or retraining.

In [17]:
%%writefile src/register.py
import argparse
import os
import mlflow

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_path", type=str, help="Path to input MLflow model")
    parser.add_argument("--model_name", type=str, default="used_cars_price_prediction_model", help="Registered model name")
    parser.add_argument("--model_info_output_path", type=str, help="Path for metadata output")
    args = parser.parse_args()

    # Load and register model into MLflow Model Registry
    model = mlflow.sklearn.load_model(args.model_path)
    registered_model = mlflow.register_model(
        model_uri=args.model_path,
        name=args.model_name
    )

    # Write output log
    os.makedirs(args.model_info_output_path, exist_ok=True)
    with open(os.path.join(args.model_info_output_path, "model_info.txt"), "w") as f:
        f.write(f"Model Name: {registered_model.name}\nVersion: {registered_model.version}")

if __name__ == "__main__":
    main()

Writing src/register.py


#### **Define Model Register Job**

For this AzureML job, a `command` object is defined to execute the `model_register.py` script. It accepts the best-trained model as input, runs the script in the `AzureML-sklearn-1.0-ubuntu20.04-py38-cpu` environment, and uses the same compute cluster as the previous jobs (`cpu-cluster`). This job plays a crucial role in the pipeline by ensuring that the best-performing model identified during hyperparameter tuning is systematically stored and made available in the MLflow registry for further evaluation, deployment, or retraining. Integrating this job into the end-to-end pipeline automates the process of registering high-quality models, completing the model development lifecycle and enabling the prediction of used car prices.

In [18]:
register_job = command(
    name="register_model",
    display_name="Model Registration Step",
    code="./src",
    command="python register.py --model_path ${{inputs.model_path}} --model_name ${{inputs.model_name}} --model_info_output_path ${{outputs.model_info_output_path}}",
    environment="machine_learning_E2E@latest",
    inputs={
        "model_path": Input(type=AssetTypes.MLFLOW_MODEL),
        "model_name": "used_cars_price_prediction_model",
    },
    outputs={
        "model_info_output_path": Output(type=AssetTypes.URI_FOLDER),
    },
    compute=cpu_compute_target,
)

### **2.4. Assembling the End-to-End Workflow**

The end-to-end pipeline integrates all the previously defined jobs into a seamless workflow, automating the process of data preparation, model training, hyperparameter tuning, and model registration. The pipeline is designed using Azure Machine Learning's `@pipeline` decorator, specifying the compute target and providing a detailed description of the workflow.

In [21]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

# Reference explicit version 6 to avoid calling the latest-version lookup API
env_ref = "machine_learning_E2E:6"

prep_job = command(
    name="prep_data",
    display_name="Data Preparation Step",
    code="./src",
    command="python prep.py --raw_data ${{inputs.raw_data}} --test_train_ratio ${{inputs.test_train_ratio}} --train_data ${{outputs.train_data}} --test_data ${{outputs.test_data}}",
    environment=env_ref,
    inputs={
        "raw_data": Input(type=AssetTypes.URI_FILE, path="azureml:used-cars-data:8"), # Explicit data version
        "test_train_ratio": 0.2,
    },
    outputs={
        "train_data": Output(type=AssetTypes.URI_FOLDER),
        "test_data": Output(type=AssetTypes.URI_FOLDER),
    },
    compute=cpu_compute_target,
)

train_job = command(
    name="train_model",
    display_name="Model Training Step",
    code="./src",
    command="python train.py --train_data ${{inputs.train_data}} --test_data ${{inputs.test_data}} --n_estimators ${{inputs.n_estimators}} --max_depth ${{inputs.max_depth}} --model_output ${{outputs.model_output}}",
    environment=env_ref,
    inputs={
        "train_data": Input(type=AssetTypes.URI_FOLDER),
        "test_data": Input(type=AssetTypes.URI_FOLDER),
        "n_estimators": 100,
        "max_depth": 10,
    },
    outputs={
        "model_output": Output(type=AssetTypes.MLFLOW_MODEL),
    },
    compute=cpu_compute_target,
)

register_job = command(
    name="register_model",
    display_name="Model Registration Step",
    code="./src",
    command="python register.py --model_path ${{inputs.model_path}} --model_name ${{inputs.model_name}} --model_info_output_path ${{outputs.model_info_output_path}}",
    environment=env_ref,
    inputs={
        "model_path": Input(type=AssetTypes.MLFLOW_MODEL),
        "model_name": "used_cars_price_prediction_model",
    },
    outputs={
        "model_info_output_path": Output(type=AssetTypes.URI_FOLDER),
    },
    compute=cpu_compute_target,
)

In [22]:
from azure.ai.ml.dsl import pipeline

@pipeline(
    compute=cpu_compute_target,
    description="Automated MLOps Pipeline for Used Car Pricing Model",
)
def car_pricing_pipeline(raw_data_input):
    prep_step = prep_job(raw_data=raw_data_input, test_train_ratio=0.2)

    train_step = train_job(
        train_data=prep_step.outputs.train_data,
        test_data=prep_step.outputs.test_data,
        n_estimators=100,
        max_depth=10,
    )

    register_step = register_job(
        model_path=train_step.outputs.model_output,
        model_name="used_cars_price_prediction_model",
    )

    return {
        "train_data": prep_step.outputs.train_data,
        "test_data": prep_step.outputs.test_data,
        "model_info": register_step.outputs.model_info_output_path,
    }

# Pass exact version 8 of data asset
pipeline_job = car_pricing_pipeline(
    raw_data_input=Input(type=AssetTypes.URI_FILE, path="azureml:used-cars-data:8")
)

# Submit pipeline job
submitted_job = ml_client.jobs.create_or_update(
    pipeline_job, 
    experiment_name="used_car_pricing_pipeline_exp",
    skip_validation=True
)

print(f"Pipeline submitted successfully!")
print(f"Studio Web URL: {submitted_job.studio_url}")
      

HttpResponseError: (AuthorizationFailed) The client 'Komal_1776179255309@npglazure.onmicrosoft.com' with object id '8ee8a280-8cd6-49f0-9877-03dfb5548bc1' does not have authorization to perform action 'Microsoft.MachineLearningServices/workspaces/codes/versions/read' over scope '/subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourceGroups/default_resourse_group/providers/Microsoft.MachineLearningServices/workspaces/demo_workspace/codes/daab90df-bb9d-4bf8-b944-131ac392f3b2' or the scope is invalid. If access was recently granted, please refresh your credentials.
Code: AuthorizationFailed
Message: The client 'Komal_1776179255309@npglazure.onmicrosoft.com' with object id '8ee8a280-8cd6-49f0-9877-03dfb5548bc1' does not have authorization to perform action 'Microsoft.MachineLearningServices/workspaces/codes/versions/read' over scope '/subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourceGroups/default_resourse_group/providers/Microsoft.MachineLearningServices/workspaces/demo_work